In [3]:
# Step 0 — Install / import libraries
import pandas as pd
import numpy as np
import nltk
import re
import string

from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Download NLTK resources
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [2]:
import pandas as pd

# Step 1 — Load Dataset
# Upload Tweets.csv from Kaggle dataset: https://www.kaggle.com/datasets/crowdflower/twitter-airline-sentiment

df = pd.read_csv("Tweets.csv")

# Check initial shape
print("Dataset shape (before dropping columns):", df.shape)
df.head()

Dataset shape (before dropping columns): (14640, 15)


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [4]:
# Step 2 — Keep only relevant columns
df = df[["airline_sentiment", "text"]]
df.head()

,airline_sentiment,text
0,neutral,@VirginAmerica What @dhepburn said.
1,positive,@VirginAmerica plus you've added commercials t...
2,neutral,@VirginAmerica I didn't today... Must mean I n...
3,negative,@VirginAmerica it's really aggressive to blast...
4,negative,@VirginAmerica and it's a really big bad thing...


In [6]:
# Step 3 — Preprocess Text
ps = PorterStemmer()

# Download punkt_tab resource if not already downloaded
nltk.download('punkt_tab', quiet=True)

def clean_text(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http.?://[^\ s]+[\ s]?', '', text)

    # Tokenize
    text = nltk.word_tokenize(text)

    # Remove stopwords
    y = []
    for i in text:
        if i not in stopwords.words('english'):
            y.append(i)
    text = y[:]
    y.clear()

    # Stemming
    for i in text:
        y.append(ps.stem(i))

    return " ".join(y)

# Apply function to create new column
df["text_cleaned"] = df["text"].apply(clean_text)
df.head()

,airline_sentiment,text,text_cleaned
0,neutral,@VirginAmerica What @dhepburn said.,@ virginamerica @ dhepburn said .
1,positive,@VirginAmerica plus you've added commercials t...,@ virginamerica plu 've ad commerci experi ......
2,neutral,@VirginAmerica I didn't today... Must mean I n...,@ virginamerica n't today ... must mean need t...
3,negative,@VirginAmerica it's really aggressive to blast...,@ virginamerica 's realli aggress blast obnoxi...
4,negative,@VirginAmerica and it's a really big bad thing...,@ virginamerica 's realli big bad thing


In [7]:
# Step 4 — Feature Extraction (TF-IDF)
tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df["text_cleaned"]).toarray()
Y = df["airline_sentiment"].values

print("Shape of X:", X.shape)
print("Shape of Y:", Y.shape)

Shape of X: (14640, 3000)
Shape of Y: (14640,)


In [8]:
# Step 5 — Split Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=2
)
print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (11712, 3000)
Test size: (2928, 3000)


In [9]:
# Step 6a — Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

nb_accuracy = accuracy_score(y_test, y_pred_nb)
print("Multinomial Naive Bayes Accuracy:", nb_accuracy)

Multinomial Naive Bayes Accuracy: 0.7223360655737705


In [10]:
# Step 6b — Random Forest Classifier
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.7513661202185792


In [11]:
# Step 7 — Extra checks (Optional)
# Count negative, neutral, positive rows
print("Sentiment counts:\n", df["airline_sentiment"].value_counts())

# Count unique neutral tweets
print("Unique neutral tweets:", df[df["airline_sentiment"]=="neutral"]["text"].nunique())

Sentiment counts:
 airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64
Unique neutral tweets: 3067


In [12]:
# After preprocessing and TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["text_cleaned"]).toarray()

# Check shape
print("Shape of X:", X.shape)

Shape of X: (14640, 3000)


In [13]:
df["airline_sentiment"].value_counts()

,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


In [14]:
# Assuming X, Y, X_train, X_test, y_train, y_test are already defined from previous preprocessing

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Train Random Forest
rf_model = RandomForestClassifier()
rf_model.fit(X_train, y_train)

# Predict on test set
y_pred_rf = rf_model.predict(X_test)

# Accuracy
rf_accuracy = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", rf_accuracy)
print("Rounded Accuracy:", round(rf_accuracy, 2))

Random Forest Accuracy: 0.7530737704918032
Rounded Accuracy: 0.75


In [15]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Assuming X_train, X_test, y_train, y_test are already defined
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Predict on test set
y_pred_nb = nb_model.predict(X_test)

# Accuracy
nb_accuracy = accuracy_score(y_test, y_pred_nb)
print("Multinomial Naive Bayes Accuracy:", nb_accuracy)
print("Rounded Accuracy:", round(nb_accuracy, 2))

Multinomial Naive Bayes Accuracy: 0.7223360655737705
Rounded Accuracy: 0.72


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df["text_cleaned"]).toarray()

In [18]:
# Count the number of rows for each sentiment
df["airline_sentiment"].value_counts()

,count
airline_sentiment,
negative,9178
neutral,3099
positive,2363


In [19]:
# Filter neutral tweets and count unique text
neutral_unique = df[df["airline_sentiment"]=="neutral"]["text"].nunique()
print("Unique neutral tweets:", neutral_unique)

Unique neutral tweets: 3067
